# 📊 ACFX: Causal Feature Exploration on German credit dataset

Welcome to this Jupyter Notebook showcasing **ACFX (Automated Causal Feature Exploration)** for the purpose of a ACFX survey [LINK].
For the purpose of this survey, I prepared and attached an examplary dataset (German credit data is a dataset by Hofmann, H. (1994). Statlog (German Credit Data) UCI Machine Learning Repository. https://doi.org/10.24432/C5NC77).


## 🔍 Purpose of this Notebook

The goal of this notebook is to:
- **Load and preprocess** the datasets.
- **Split** into training and testing sets with stratification for reproducibility.
- **Explore causal relationships** among features using ACFX.
- **Visualize causal graphs** to uncover dependencies and potential actionable insights.
- Enable you to **answer questions of the survey** regarding your opinions of the framework.

By the end of this notebook, you will be able to:

- Identify **key causal relationships** in the data.  
- Understand **feature importance from a causal perspective**, not just correlation.  
- Generate **graphical visualizations** of the learned causal structures.  
- Use ACFX framework with bayesian network causability model
## Prelimiary
- Please review the prepared dataset loaded from [_german_credit_numeric.csv_](https://raw.githubusercontent.com/sbobek/acfx/users/pku/student-study/dataset/german_credit_numeric.csv). Detailed descriptions of each feature are available in the legend section (below). Feature types have been assigned according to the mapping defined in _column_types_ dictionary (below)
---

# German credit Dataset legend

**Attribute 1:** (qualitative)
 Status of existing checking account
             0 :      ... <    0 DM
	       1 : 0 <= ... <  200 DM
	       2 :      ... >= 200 DM / salary assignments for at least 1 year
               3 : no checking account

**Attribute 2:** (numerical)
	      Duration in month

**Attribute 3:**  (qualitative)
	      Credit history
	      0 : no credits taken/ all credits paid back duly
          1 : all credits at this bank paid back duly
	      2 : existing credits paid back duly till now
          3 : delay in paying off in the past
	      4 : critical account/  other credits existing (not at this bank)

**Attribute 4:**  (qualitative)
	      Purpose
	      0 : car (new)
	      1 : car (used)
	      2 : furniture/equipment
	      3 : radio/television
	      4 : domestic appliances
	      5 : repairs
	      6 : education
	      7 : (vacation - does not exist?)
	      8 : retraining
	      9 : business
	      10 : others

**Attribute 5:**  (numerical)
	      Credit amount

**Attibute 6:**  (qualitative)
	      Savings account/bonds
	      0 :          ... <  100 DM
	      1 :   100 <= ... <  500 DM
	      2 :   500 <= ... < 1000 DM
	      3 :          .. >= 1000 DM
          4 :   unknown/ no savings account

**Attribute 7:**  (qualitative)
	      Present employment since
	      0 : unemployed
	      1 :       ... < 1 year
	      2 : 1  <= ... < 4 years
	      3 : 4  <= ... < 7 years
	      4 :       .. >= 7 years

**Attribute 8:**  (numerical)
	      Installment rate in percentage of disposable income

**Attribute 9:** (qualitative)
	      Personal status and sex
	      0 : male   : divorced/separated
	      1 : female : divorced/separated/married
          2 : male   : single
	      3 : male   : married/widowed
	      4 : female : single

**Attribute 10:** (qualitative)
	      Other debtors / guarantors
	      0 : none
	      1 : co-applicant
	      2 : guarantor

**Attribute 11:** (numerical)
	      Present residence since

**Attribute 12:** (qualitative)
	      Property
	      0 : real estate
	      1 : if not 0 : building society savings agreement/ life insurance
          2 : if not 0/1 : car or other, not in attribute 6
	      3 : unknown / no property

**Attribute 13:** (numerical)
	      Age in years

**Attribute 14:** (qualitative)
	      Other installment plans
	      0 : bank
	      1 : stores
	      2 : none

**Attribute 15:** (qualitative)
	      Housing
	      0 : rent
	      1 : own
	      2 : for free

**Attribute 16:** (numerical)
              Number of existing credits at this bank

**Attribute 17:** (qualitative)
	      Job
	      0 : unemployed/ unskilled  - non-resident
	      1 : unskilled - resident
	      2 : skilled employee / official
	      3 : management/ self-employed/
		     highly qualified employee/ officer

**Attribute 18:** (numerical)
	      Number of people being liable to provide maintenance for

**Attribute 19:** (qualitative)
	      Telephone
	      0 : none
	      1 : yes, registered under the customers name

**Attribute 20:** (qualitative)
	      foreign worker
	      0 : yes
	      1 : no

**Target:** (qualitative)
	      Credit granted
	      0   : no
	      1   : yes


In [ ]:
!python --version

In [ ]:
!pip install acfx==0.3.6

In [ ]:
!pip install ipywidgets pydot

In [ ]:
!pip install pyvis

In [ ]:
# utils
import numpy as np
import pandas as pd

# source code by Szymon Bobek: https://colab.research.google.com/drive/1Hj6yH4UIrAp1Jp6B1U542vcdSkXuZHUd (accessed and modified: 5 May 2026)
def make_counterfactual_delta_table(
    query_df: pd.DataFrame,
    cf_df: pd.DataFrame,
    decimals: int = 3,
    tol: float = 1e-12,
    cf_index_prefix: str = "CF #",
    feature_types:dict[str,str]=None
):
    """
    Build a styled DataFrame indicating how much each feature should change
    to obtain each counterfactual, with signed formatting like +0.232.

    Parameters
    ----------
    query_df : pd.DataFrame
        The original instance(s), unscaled. If it contains 1 row, it will be broadcast
        to the number of rows in cf_df.
    cf_df : pd.DataFrame
        The generated counterfactual instance(s), unscaled. Must have the same columns as query_df.
        Each row is a distinct counterfactual.
    decimals : int
        Number of decimal places to display for deltas.
    tol : float
        Absolute tolerance under which a delta is considered zero (displayed as blank).
    cf_index_prefix : str
        Prefix for counterfactual row names.

    Returns
    -------
    styled : pd.io.formats.style.Styler
        A styled table with signed deltas and color cues (green for increases, red for decreases).
    delta_df : pd.DataFrame
        The numeric delta DataFrame (cf_df - query_df).
    """

    # --- Basic validation
    if not isinstance(query_df, pd.DataFrame) or not isinstance(cf_df, pd.DataFrame):
        raise TypeError("query_df and cf_df must be pandas DataFrames")

    if list(query_df.columns) != list(cf_df.columns):
        raise ValueError("query_df and cf_df must have identical columns (same order).")

    # Broadcast query row if a single instance is provided
    if len(query_df) == 1 and len(cf_df) > 1:
        query_aligned = pd.DataFrame(
            np.repeat(query_df.values, repeats=len(cf_df), axis=0),
            columns=query_df.columns,
            index=cf_df.index
        )
    else:
        # Otherwise, they must have the same number of rows
        if len(query_df) != len(cf_df):
            raise ValueError(
                f"Row mismatch: query_df has {len(query_df)} rows, cf_df has {len(cf_df)} rows. "
                f"Provide one query row to broadcast or match the counts."
            )
        query_aligned = query_df.copy()
        query_aligned.index = cf_df.index

    # Compute deltas (what to add to the original to reach the CF)
    delta = cf_df - query_aligned

    # Index: label rows as CF #1, CF #2, ...
    delta.index = [f"{cf_index_prefix}{i+1}" for i in range(len(delta))]

    # Format as signed strings (blank if ~0)
    def _fmt_signed(x,col_name:str):
        if pd.isna(x):
            return ""
        if abs(x) < tol:
            return ""
        if feature_types is not None and col_name in feature_types.keys() \
            and (feature_types[str(col_name)] == "ordinal" or feature_types[str(col_name)] == "nominal"):
            s = f"{int(x):+d}"
        else:
            s = f"{x:+.{decimals}f}"
        return s

    formatted = delta.apply(lambda col: col.map(lambda val: _fmt_signed(val,col.name)))

    # Styling: green for positive, red for negative, gray for zero/blank
    def color_changes(val):
        if val == "":
            return "color: #888888;"  # gray for (near-)zero
        return "color: #2e7d32;" if val.strip().startswith("+") else "color: #c62828;"

    styled = (
        formatted.style
        .applymap(color_changes)
        .set_properties(**{
            "font-family": "Segoe UI, Roboto, Arial, sans-serif",
            "font-size": "12.5px",
            "white-space": "nowrap",
        })
        .set_table_styles([
            {"selector": "th", "props": [("font-weight", "600"), ("text-align", "center")]},
            {"selector": "td", "props": [("text-align", "center"), ("padding", "6px 10px")]},
        ])
        .set_caption("Required feature adjustments to obtain each counterfactual (unscaled)")
    )

    return styled, delta


def counterfactual_instructions(delta_df: pd.DataFrame, tol: float = 1e-12, decimals: int = 3, feature_types: dict[str,str] = None):
    """
    Build natural-language instructions for each counterfactual row:
    Example: 'Increase X by 0.232; decrease Y by 1.50'

    Returns
    -------
    dict: {row_label: instruction_str}
    """
    instructions = {}
    for idx, row in delta_df.iterrows():
        parts = []
        for col, v in row.items():
            if pd.isna(v) or abs(v) < tol:
                continue
            direction = "increase" if v > 0 else "decrease"
            if feature_types is not None and col in feature_types.keys() and (feature_types[str(col)] == "ordinal" or feature_types[str(col)] == "nominal"):
                parts.append(f"{direction} {col} by {abs(int(v))}")
            else:
                parts.append(f"{direction} {col} by {abs(v):.{decimals}f}")
        instructions[idx] = "; ".join(parts) if parts else "No changes needed"
    return instructions


In [ ]:
feature_types ={
  "Status of existing checking account": "ordinal",
  "Duration":"continuous" ,
  "Credit history":"ordinal" ,
  "Purpose":"nominal" ,
  "Credit amount": "continuous",
  "Savings account/bonds": "ordinal",
  "Present employment since": "ordinal",
  "Installment rate in percentage of disposable income": "continuous",
  "Personal status and sex": "nominal",
  "Other debtors / guarantors": "nominal",
  "Present residence since": "continuous",
  "Property": "nominal",
  "Age": "continuous",
  "Other installment plans": "nominal",
  "Housing": "nominal",
  "Number of existing credits at this bank": "ordinal",
  "Job": "ordinal",
  "Number of people being liable to provide maintenance for": "ordinal", # there are only two options: 1 and 2
  "Telephone": "nominal",
  "foreign worker": "nominal",
  "Credit granted": "nominal",
}

In ACFX framework, there are three explainers available for evaluation:
1. AcfxEBM, for interpret.glassbox's ExplainableBoostingClassifier blackbox
2. AcfxLinear, for scikit-learn's LinearClassifierMixin (linear classifier with linear coeffs- LogisticRegressionCounterOptimizer is recommended here)
3. AcfxCustom, for the use of any custom blackbox (requires providing compatible counter-optimizer)

For the purpose of this survey, we will stick to the EBM blackbox and the german credit dataset

To initialize the explainer with its blackbox, run:

In [ ]:
from acfx import AcfxEBM
from interpret.glassbox import ExplainableBoostingClassifier

no_target_feature_types = [
    v for k, v in feature_types.items() if k != 'Credit granted'
]

categorical_indicator = [
    True if v != 'continuous' else False
    for k, v in feature_types.items()
    if k != 'Credit granted'
]

model = ExplainableBoostingClassifier(
    # Can be: 'continuous', 'nominal', or 'ordinal'
    feature_types=no_target_feature_types)
explainer = AcfxEBM(model)

In [ ]:
import pandas as pd
file_path = 'https://raw.githubusercontent.com/sbobek/acfx/users/pku/student-study/dataset/german_credit_numeric.csv'

In [ ]:
bunch_df = pd.read_csv(file_path)

In [ ]:
bunch_df

In [ ]:
X = bunch_df.drop('Credit granted', axis=1)
y = bunch_df['Credit granted']

In [ ]:
# You can use seed to fix the train-test split
# SEED = 43

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.1,
# random_state=SEED
)

In [ ]:
%%capture --no-display
import logging
logging.getLogger("interpret").setLevel(logging.WARNING)
model.fit(X=X_train, y=y_train)

We will now train a pgmpy's DiscreteBayesianNetwork model that operates on discrete features. Continuous features will be discretized.

In [ ]:
%%capture --no-display
import logging
logging.getLogger("pgmpy").setLevel(logging.WARNING)

from acfx.evaluation.bayesian_model import train_bayesian_model
num_bins = 3
bayesian_model = train_bayesian_model(X_train, categorical_indicator, num_bins)


We will now display the DAG, which is generated based on the causability obtained from the Bayesian network model

In [ ]:
nodes = list(bayesian_model.nodes)
edges = list(bayesian_model.edges)

In [ ]:
import networkx as nx
G = nx.DiGraph()
G.add_nodes_from(nodes)
G.add_edges_from(edges)

In [ ]:
from pyvis.network import Network
import networkx as nx

net = Network(notebook=True, directed=True, height="600px", width="100%", bgcolor="#222222", font_color="white")

net.from_nx(G)
net.toggle_physics(True)
net.show_buttons(filter_=['physics'])
net.show("bayesian_network.html")

In [ ]:
causal_order = list(nx.topological_sort(G))
print("Causal order:", causal_order)

We will now loop through the CPDs (conditional probability distributions) to display extracted features' causability model

In [ ]:
from IPython.display import display, Markdown
limit = 3

i=0
for cpd in bayesian_model.get_cpds():
    # comment to display all
    if limit == i:
        break

    if not cpd:
        continue
    try:
        df = cpd.to_dataframe()

        display(Markdown("---"))

        display(Markdown(f"### CPD of Variable: **{cpd.variable}**"))

        evidence = cpd.get_evidence()
        if evidence:
            display(Markdown(f"*Conditioned on (Evidence):* `{', '.join(evidence)}`"))
        else:
            display(Markdown(f"*Prior Probability (Root Node - No Evidence)*"))

        display(df)
        i+=1
    except Exception as e:
        continue

In [ ]:
pbounds = {col: (X_train[col].min(), X_train[col].max()) for col in X_train.columns}
features_order = X_train.columns.tolist()

query_instance = X_test.sample(1).values

We will now fit the counterfactual explainer in order to showcase counterfactual generation process. For your convenience, counterfactual generation parameters are displayed in the cell below. You can try and manipulate them to get different results



In [ ]:

# What other features would you make not available to change?
fixed = {
    'Number of people being liable to provide maintenance for',
    'foreign worker',
    'Personal status and sex',
    "Credit amount",
    'Age'
}

PLAUSABILITY_WEIGHT=0.9
DIVERSITY_WEIGHT=0.1
SPARSITY_WEIGHT=0.1
INIT_POINTS=100

masked_features = [x for x in features_order if x not in fixed]

In [ ]:
explainer.fit(X=X_train,
              pbounds=pbounds,
              masked_features=masked_features,
              categorical_indicator=categorical_indicator,
              features_order=features_order,
              bayesian_causality=True,
              num_bins=num_bins,
              bayesian_model=bayesian_model)

In [ ]:
original_class = model.predict([query_instance])[0]
desired_class = 1 if int(original_class) == 0 else 0

In [ ]:
cf = explainer.counterfactual(desired_class=desired_class, query_instance=query_instance,plausibility_weight=PLAUSABILITY_WEIGHT, diversity_weight=DIVERSITY_WEIGHT, sparsity_weight=SPARSITY_WEIGHT,init_points=INIT_POINTS)

Counterfactual is generated. Let's see what are the proposed changes

In [ ]:
import pandas as pd

# source code by Szymon Bobek: https://colab.research.google.com/drive/1Hj6yH4UIrAp1Jp6B1U542vcdSkXuZHUd (accessed and modified: 5 May 2026)
feature_names = X_train.columns.tolist()  # or your saved feature names
query_df = pd.DataFrame(query_instance, columns=feature_names)
cf_df = pd.DataFrame(cf, columns=feature_names)

comparison_df = pd.concat([query_df, cf_df], keys=['Original', 'Counterfactual'])

display(comparison_df)


styled, delta_df = make_counterfactual_delta_table(query_df, cf_df, decimals=3, feature_types=feature_types)

try:
    from IPython.display import display
    display(styled)
except Exception:
    print("\nRequired adjustments (delta = CF - Original):")
    print(delta_df.to_string(float_format=lambda x: f"{x:+.3f}" if abs(x) > 1e-12 else " 0.000"))

instr = counterfactual_instructions(delta_df, decimals=3, feature_types=feature_types)
print("\nInstructions:")
for k, v in instr.items():
    print(f"  {k}: {v}")
